<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/SaliAI_DeepPrompt_Fusion_Track2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — Guidance-Rich Prompt + Fusion (ALLaM & Fanar) — Track 2

Applying the deep guidance-rich prompt to Track 2 (Electric-Cars / Trimester).
Reference: Fanar = 0.828 (best) | ALLaM ~ 0.70. Fusion may hurt (models differ in
strength), but we try it and document the result.

Input: `test_unseen.csv`. Requires a T4 GPU.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.7 MB/s eta 0:00:00


In [ ]:
import torch, os, re, zipfile
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd
assert torch.cuda.is_available(), "فعّل GPU"
print("GPU:", torch.cuda.get_device_name(0))
DATA_DIR="./data"; OUT="./deep_prompt_out"; os.makedirs(OUT, exist_ok=True)
TEXT_COL="text"; TARGET_COL="target"
df=pd.read_csv(f"{DATA_DIR}/test_unseen.csv", keep_default_na=False)
if "tweet_text" in df.columns and TEXT_COL not in df.columns: df=df.rename(columns={"tweet_text":TEXT_COL})
print("اختبار Track 1:", len(df))
TARGET_DESC={"Ecars":"السيارات الكهربائية والتحول إليها","Trimester":"نظام الفصول الدراسية الثلاثة في التعليم"}
def desc(t):
    t=str(t).strip()
    return TARGET_DESC.get(t,t)


GPU: Tesla T4
اختبار Track 1: 644


In [ ]:
# Guidance-rich prompt (rich guidance + direct answer)
SYS_DEEP=(
"أنت خبير لغوي عربي متمرّس في تحليل المواقف على وسائل التواصل. "
"مهمتك: تحديد موقف كاتب التغريدة الحقيقي تجاه الهدف. راعِ ما يلي بدقة قبل أن تقرر:\n"
"- افهم اللهجة (خليجية/حجازية/نجدية) والتعابير العامية والسخرية والتهكّم.\n"
"- ميّز بين رأي الكاتب نفسه، ورأيٍ يقتبسه أو يسخر منه أو يردّ عليه (قد يكون موقفه عكس ظاهر الكلمات أو الوسم).\n"
"- استشعر نبرة الكاتب وشعوره لحظة الكتabة (غضب، تأييد، تهكّم، حياد).\n"
"- الوسوم (#) تدل على موضوع النقاش لا على موقف الكاتب بالضرورة.\n"
"ثم أجب بكلمة إنجليزية واحدة فقط دون أي شرح: Favor أو Against أو None. "
"Favor إن كان مؤيداً للهدف، Against إن كان معارضاً، None إن لم يظهر موقف شخصي واضح.")

FEW=[("التطعيم أنقذ ملايين الأرواح ولازم الكل ياخذه","لقاح كورونا","Favor"),
     ("ما أثق باللقاح وله أضرار كثيرة، مؤامرة","لقاح كورونا","Against"),
     ("متى تفتح مراكز التطعيم؟","لقاح كورونا","None")]

def build(text,target):
    m=[{"role":"system","content":SYS_DEEP}]
    for tw,tg,l in FEW:
        m+=[{"role":"user","content":f"الهدف: {tg}\nالتغريدة: {tw}\nالموقف:"},{"role":"assistant","content":l}]
    m.append({"role":"user","content":f"الهدف: {target} ({desc(target)})\nالتغريدة: {text}\nالموقف:"})
    return m
def parse(o):
    o=o.strip().lower()
    poss=[(o.rfind("against"),"Against"),(o.rfind("favor"),"Favor"),(o.rfind("none"),"None"),
          (o.rfind("معارض"),"Against"),(o.rfind("مؤيد"),"Favor"),(o.rfind("محايد"),"None")]
    poss=[(i,l) for i,l in poss if i>=0]
    return max(poss)[1] if poss else "None"

In [ ]:
# Model loading + prediction function
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

def run_model(model_id, tag, use_vote=False, n_vote=5):
    print(f"\n== {tag} (vote={use_vote}) ==")
    tok=AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    mdl=AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    mdl.eval()
    def gen(msgs, sample=False, temp=0.7):
        p=tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        e=tok(p, return_tensors="pt", return_token_type_ids=False).to(mdl.device)
        with torch.no_grad():
            o=mdl.generate(**e, max_new_tokens=6, do_sample=sample,
                           temperature=temp if sample else None, top_p=0.95 if sample else None,
                           pad_token_id=tok.eos_token_id)
        return parse(tok.decode(o[0][e["input_ids"].shape[-1]:], skip_special_tokens=True))
    preds=[]
    temps=[0.3,0.5,0.7,0.9,0.6]
    for _,r in tqdm(df.iterrows(), total=len(df), desc=tag):
        t=str(r[TEXT_COL]); tg=str(r[TARGET_COL]); m=build(t,tg)
        if use_vote:
            votes=[gen(m, sample=True, temp=temps[k%len(temps)]) for k in range(n_vote)]
            preds.append(Counter(votes).most_common(1)[0][0])
        else:
            preds.append(gen(m, sample=False))
    print(f"توزيع {tag}:", Counter(preds))
    del mdl; gc.collect(); torch.cuda.empty_cache()
    return preds

def save_sub(preds, name):
    txt=f"{OUT}/{name}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    with zipfile.ZipFile(f"{OUT}/{name}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
    print(f"  saved {name}")

In [ ]:
# ALLaM (deep + vote) and Fanar (deep + direct)
allam = run_model("humain-ai/ALLaM-7B-Instruct-preview", "allam_deep_vote", use_vote=True, n_vote=5)
save_sub(allam, "sub_allam_t2_deep_vote")
fanar = run_model("QCRI/Fanar-1-9B-Instruct", "fanar_deep_direct", use_vote=False)
save_sub(fanar, "sub_fanar_t2_deep_direct")


== allam_deep_vote (vote=True) ==


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

fanar_deep_direct:   0%|          | 0/644 [00:00<?, ?it/s]

توزيع fanar_deep_direct: Counter({'Against': 371, 'Favor': 205, 'None': 68})
  saved sub_fanar_t2_deep_direct


In [ ]:
# Fusion (ALLaM stronger -> priority on disagreement, avoid None)
n=min(len(allam),len(fanar)); a=allam[:n]; f=fanar[:n]
agree=sum(1 for i in range(n) if a[i]==f[i])
print(f"اتفاق ALLaM/Fanar (عميق): {agree}/{n} ({agree/n*100:.1f}%)")
fused=[]
for i in range(n):
    x,y=a[i],f[i]
    if x==y: fused.append(x)
    elif x=="None": fused.append(y)
    elif y=="None": fused.append(x)
    else: fused.append(x)   # رجّح ALLaM (الأقوى على Track 1)
save_sub(fused, "sub_t2_deep_fusion")
print("توزيع الدمج العميق:", Counter(fused))
print("\n>>> ارفع sub_allam_t2_deep_vote + sub_fanar_t2_deep_direct + sub_t2_deep_fusion وقارن مع 0.7428 <<<")

اتفاق ALLaM/Fanar (عميق): 389/644 (60.4%)
  saved sub_t2_deep_fusion
توزيع الدمج العميق: Counter({'Against': 309, 'Favor': 283, 'None': 52})

>>> ارفع sub_allam_t2_deep_vote + sub_fanar_t2_deep_direct + sub_t2_deep_fusion وقارن مع 0.7428 <<<
